In [ ]:
import os
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib
from adjustText import adjust_text
import scipy.stats as stats
import numpy as np
#from matplotlib.colors import LogNorm
#import matplotlib.ticker as ticker

file = os.path.join('datasets', 'sarko_affaires.json')
file2 = os.path.join('datasets', 'sarko_title.json')
file3 = os.path.join('datasets', 'sarko_all.json')

SARKORPUS_LIGHT

By newspaper (Libération, Le Monde, Le Figaro):

In [36]:
def graph_j3(path, threshold=1000):
    dic = {1: [], 2: [], 3: []}
    n_mots = 0
    df = pd.read_csv(path, encoding='utf8', sep=';')
    for w in df.itertuples():
            if n_mots < threshold:
                if w[5] > 0.0:
                    n_mots += 1
                    for j in range(1,4):
                        dic[j].append(w[j+1])
            else:
                break
    print("Nombre mots analysés:", n_mots)

    data = [dic[1], dic[2], dic[3]]
    plt.figure(figsize=(10,6))
    plt.boxplot(data, showfliers=False, showmeans=True, meanline=True, meanprops={"color": "red"})
    # Calcul des moyennes et médianes
    means = [np.mean(d) for d in data]
    medians = [np.median(d) for d in data]
    # Annotation des valeurs
    for i, (mean, median) in enumerate(zip(means, medians), start=1):
        # Moyenne (rouge)
        plt.text(i, mean, f"μ={mean:.3f}",
                 ha='center', va='bottom',
                 color='red', fontsize=10)
        # Médiane (bleu)
        plt.text(i, median, f"m={median:.3f}",
                 ha='center', va='top',
                 color='blue', fontsize=10)
    #plt.title("Dissimilarity by newspapers")
    plt.xlabel("Newspapers compared")
    plt.ylabel("Dissimilarity")
    plt.gca().xaxis.set_ticklabels(['Libération->Le Monde', 'Libération->Le Figaro', 'Le Monde->Le Figaro'])
    plt.plot()

In [ ]:
path = os.path.join('by_newspapers', 'results', 'results_pretrained_selected_words.csv')
graph_j3(path, 1000)

By period (political before the 15/05/2012 VS judicial after the 15/05/2012 - for Le Figaro, Libération, Le Monde)

In [ ]:
def graph_period(threshold=1000):
    dic = {'intra political': [], 'intra judicial': [], 'extra': []}
    n_mots = 0
    path = os.path.join('by_period', 'since_2002', 'results', 'results_pretrained_selected_words.csv')
    df = pd.read_csv(path, encoding='utf8', sep=';')
    for w in df.itertuples():
            if n_mots < threshold:
                if w[8] > 0.0:
                    n_mots += 1
                    dic['intra political'].append(w[2])
                    dic['intra judicial'].append(w[3])
                    for i in range(4,8):
                         dic['extra'].append(w[i])
            else:
                break
    print("Nombre mots analysés:", n_mots)

    data = [dic['intra political'], dic['intra judicial'], dic['extra']]
    plt.figure(figsize=(10,6))
    plt.boxplot(data, showfliers=False, showmeans=True, meanline=True, meanprops={"color": "red"})
    # Calcul des moyennes et médianes
    means = [np.mean(d) for d in data]
    medians = [np.median(d) for d in data]
    # Annotation des valeurs
    for i, (mean, median) in enumerate(zip(means, medians), start=1):
        # Moyenne (rouge)
        plt.text(i, mean, f"μ={mean:.3f}",
                 ha='center', va='bottom',
                 color='red', fontsize=10)
        # Médiane (bleu)
        plt.text(i, median, f"m={median:.3f}",
                 ha='center', va='top',
                 color='blue', fontsize=10)
    #plt.title("Dissimilarity by period")
    plt.xlabel("Periods compared")
    plt.ylabel("Dissimilarity")
    plt.gca().xaxis.set_ticklabels(['Intra-political', 'Intra-judicial', 'Political - Judicial'])
    plt.plot()

In [ ]:
graph_period(1000)

T-Test for paired samples

In [ ]:
def get_tabs_journal(path):
    dic = {1: [], 2: [], 3: []}
    df = pd.read_csv(path, encoding='utf8', sep=';')
    for w in df.itertuples():
        if w[2] > 0.0 and w[3] > 0.0 and w[4] > 0.0:
            for j in range(1,4):
                dic[j].append(w[j+1])
    return dic[1], dic[2], dic[3]

def get_tabs_periode():
    dic = {'intra political': [], 'intra judicial': [], 4: [], 5: [], 6: [], 7: []}
    path = os.path.join('by_period', 'since_2002', 'results', 'results_pretrained_selected_words.csv')
    df = pd.read_csv(path, encoding='utf8', sep=';')
    for w in df.itertuples():
        if (w[i] > 0.0 for i in range(2,8)):
            dic['intra political'].append(w[2])
            dic['intra judicial'].append(w[3])
            for j in range(4,8):
                dic[j].append(w[j])
    return dic['intra political'], dic['intra judicial'], dic[4], dic[5], dic[6], dic[7]

In [ ]:
def heatmap(data, row_labels, col_labels, ax=None, cbar_kw=None, cbarlabel="", **kwargs):
    if ax is None:
        ax = plt.gca()

    if cbar_kw is None:
        cbar_kw = {}

    # Plot the heatmap
    im = ax.imshow(data, **kwargs)

    # Create colorbar
    cbar = ax.figure.colorbar(im, ax=ax, **cbar_kw)
    cbar.ax.set_ylabel(cbarlabel, rotation=-90, va="bottom")

    # Show all ticks and label them with the respective list entries.
    ax.set_xticks(range(data.shape[1]), labels=col_labels,
                  rotation=-30, ha="right", rotation_mode="anchor")
    ax.set_yticks(range(data.shape[0]), labels=row_labels)

    # Let the horizontal axes labeling appear on top.
    ax.tick_params(top=True, bottom=False,
                   labeltop=True, labelbottom=False)

    # Turn spines off and create white grid.
    ax.spines[:].set_visible(False)

    ax.set_xticks(np.arange(data.shape[1]+1)-.5, minor=True)
    ax.set_yticks(np.arange(data.shape[0]+1)-.5, minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)

    return im, cbar

def annotate_heatmap(im, data=None, valfmt="{x:.2f}", textcolors=("black", "white"), threshold=None, **textkw):
    if not isinstance(data, (list, np.ndarray)):
        data = im.get_array()

    # Normalize the threshold to the images color range.
    if threshold is not None:
        threshold = im.norm(threshold)
    else:
        threshold = im.norm(data.max())/2.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",
              verticalalignment="center")
    kw.update(textkw)

    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = matplotlib.ticker.StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(im.norm(data[i, j]) > threshold)])
            text = im.axes.text(j, i, valfmt(data[i, j], None), **kw)
            texts.append(text)

    return texts


def pval_format(x, y):
    if x < 1e-4:
        return f"{x:.1e}"
    else:
        return f"{x:.4f}"

def pval_format2(x,y):
    if x != 1:
        return f"{x:.1e}"
    else:
        return f"{x:.0f}"

def heatmap_student():
    ref = {"political_0 / political_1": 0, "judicial_0 / judicial_1": 1, "political_0 / judicial_0": 2, "political_0 / judicial_1": 3, "political_1 / judicial_0": 4, "political_1 / judicial_1": 5}
    t1, t2, t3, t4, t5, t6 = get_tabs_periode()
    ts = [t1, t2, t3, t4, t5, t6]
    n = len(ts)
    res_s = np.zeros((n,n))
    for i in range(n):
        for j in range(n):
            """if i == j:
                p = 1
            else:"""
            test = stats.ttest_rel(ts[i], ts[j])
            p = test.pvalue
            res_s[i][j] = p
    res_s = -np.log10(res_s)
    fig, ax = plt.subplots(figsize=(8,5))
    im, cbar = heatmap(res_s, ref.keys(), ref.keys(), ax=ax, cmap="YlGn", cbarlabel="-log10(p value)") #, norm=LogNorm(vmin=res_s.min(), vmax=res_s.max()))
    #cbar.ax.yaxis.set_major_locator(ticker.LogLocator(base=10))
    #cbar.ax.yaxis.set_major_formatter(ticker.LogFormatterMathtext(base=10))
    texts = annotate_heatmap(im, valfmt="{x:.2f}")
    fig.tight_layout()
    plt.plot()

In [ ]:
heatmap_student()

In [ ]:
path = os.path.join('by_newspapers', 'results', 'results_pretrained_selected_words.csv')
t1, t2, t3 = get_tabs_journal(path)

print("By papers (3):")
print("liberation<=>monde VS liberation<=>figaro")
print(stats.ttest_rel(t1, t2))
print()
print("monde<=>liberation VS monde<=>figaro")
print(stats.ttest_rel(t1, t3))
print()
print("figaro<=>liberation VS figaro<=>monde")
print(stats.ttest_rel(t2, t3))

In [ ]:
t1, t2, t3, t4, t5, t6 = get_tabs_periode()

print("By period (before/after 15/05/2012):")
print("intra political VS intra judicial")
rel0 = stats.ttest_rel(t1, t2)
print(rel0)
print(rel0.confidence_interval())
print()
print("intra political VS extra")
rel1 = stats.ttest_rel(t1, t3)
print(rel1)
print(rel1.confidence_interval())
print(stats.ttest_rel(t1, t4))
print(stats.ttest_rel(t1, t5))
print(stats.ttest_rel(t1, t6))
print()
print("intra judicial VS extra")
print(stats.ttest_rel(t2, t3))
print(stats.ttest_rel(t2, t4))
print(stats.ttest_rel(t2, t5))
print(stats.ttest_rel(t2, t6))

SARKORPUS_BIG_TITLES

In [ ]:
path_title = os.path.join('by_newspapers_title', 'results', 'results_pretrained_selected_words.csv')
graph_j3(path_title, 1000)

In [ ]:
path = os.path.join('by_newspapers_title', 'results', 'results_pretrained_selected_words.csv')
t1, t2, t3 = get_tabs_journal(path)

print("By paper (3):")
print("liberation<=>monde VS liberation<=>figaro")
print(stats.ttest_rel(t1, t2))
print()
print("monde<=>liberation VS monde<=>figaro")
print(stats.ttest_rel(t1, t3))
print()
print("figaro<=>liberation VS figaro<=>monde")
print(stats.ttest_rel(t2, t3))

SARKORPUS_BIG all

In [ ]:
path_all = os.path.join('by_newspapers_all', 'results', 'results_pretrained_selected_words.csv')
graph_j3(path_all, 3000)

In [ ]:
path = os.path.join('by_newspapers_all', 'results', 'results_pretrained_selected_words.csv')
t1, t2, t3 = get_tabs_journal(path)

print("By paper (3):")
print("liberation<=>monde VS liberation<=>figaro")
print(stats.ttest_rel(t1, t2))
print()
print("monde<=>liberation VS monde<=>figaro")
print(stats.ttest_rel(t1, t3))
print()
print("figaro<=>liberation VS figaro<=>monde")
print(stats.ttest_rel(t2, t3))